# Fix 1, stage B: step 4 on the QQP population

**Use only if at least one PAWS setting passed the fix-1 gate (re-run with the amended, group-blocked gate).**

**Session options:** GPU T4 or P100, Internet On. **Add Input:** the output of the fixes 1+4 notebook (or `fix14_out.zip` as a dataset).

Run cells 1–9 by hand (cell 9 times 2 models), then **Save & Run All (Commit)** (about 2–3 hours). Download `qqp_step4_out.zip` from the Output tab.

In [ ]:
%%writefile feather_eval.py
#!/usr/bin/env python3
"""
Steps 2-3 of the "mechanistic vs representational predictors of OOD behaviour" project.

Population: the 100 BERT-base models McCoy et al. (2020) fine-tuned on MNLI, hosted as
    connectivity/feather_berts_0 ... connectivity/feather_berts_99   (Juneja et al. 2023)

Subcommands
-----------
  check    Step 2. Load a few models; verify (a) the label order, (b) that our forward pass
           reproduces the logits stored in each repo's mnli_eval_bert_{i}.json, and (c) that
           MNLI matched-dev accuracy falls in McCoy et al.'s reported 83.6-84.8% range.
  eval     Step 3a. Run every model on MNLI matched-dev and all 30k HANS items and save
           per-example logits. Resumable: models with an existing output file are skipped.
  analyze  Step 3b. Is the OOD outcome real? Split-half reliability and between-model spread
           of HANS lexical-overlap non-entailment accuracy, plus diagnostics. Prints PASS/FAIL
           against pre-set gates.

Typical use (on a GPU machine with Hugging Face access)
-------------------------------------------------------
  python feather_eval.py check   --models 0 1 2 3 4
  python feather_eval.py eval    --models all --purge
  python feather_eval.py analyze

Label conventions (verified against the repos' own eval files):
  model output index : 0 = contradiction, 1 = entailment, 2 = neutral   (Google BERT MnliProcessor)
  GLUE/HF MNLI label : 0 = entailment,    1 = neutral,    2 = contradiction
  HANS               : prediction is "entailment" iff argmax == 1; neutral + contradiction
                       collapse to "non-entailment" (as in McCoy et al.).
"""

import argparse
import csv
import json
import os
import platform
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- constants

REPO_TMPL = "connectivity/feather_berts_{i}"
N_MODELS = 100
MODEL_LABELS = {"contradiction": 0, "entailment": 1, "neutral": 2}
ENT = MODEL_LABELS["entailment"]
GLUE_TO_MODEL = np.array([1, 2, 0])  # GLUE ent/neu/con -> model index
HANS_URL = "https://raw.githubusercontent.com/tommccoy1/hans/master/heuristics_evaluation_set.txt"
MCCOY_MNLI_RANGE = (0.836, 0.848)  # reported MNLI matched-dev range across the 100 models
MAX_LEN = 128                       # max_seq_length used by McCoy et al. (Google BERT default)

# Pre-registered gates for step 3 (set before seeing data; do not tune).
GATE_SPLIT_HALF_R = 0.80   # raw (not Spearman-Brown) stratified split-half correlation
GATE_SD_RATIO = 3.0        # between-model SD / binomial sampling SD
GATE_RANGE = 0.10          # max - min accuracy across models


# ----------------------------------------------------------------------------- data

def load_hans(path):
    import pandas as pd
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading HANS to {path} ...")
        urllib.request.urlretrieve(HANS_URL, path)
    d = pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    assert len(d) == 30000, f"HANS should have 30000 rows, got {len(d)}"
    return {
        "premise": d["sentence1"].tolist(),
        "hypothesis": d["sentence2"].tolist(),
        "gold_ent": (d["gold_label"] == "entailment").to_numpy(),
        "heuristic": d["heuristic"].to_numpy(),
        "subcase": d["subcase"].to_numpy(),
        "template": d["template"].to_numpy(),
        "pairID": d["pairID"].to_numpy(),
    }


def load_mnli(tsv=None):
    """MNLI matched dev. Labels returned in MODEL index space."""
    if tsv:  # GLUE dev_matched.tsv
        prem, hyp, lab = [], [], []
        name2model = MODEL_LABELS
        with open(tsv, newline="", encoding="utf-8") as f:
            r = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
            for row in r:
                if row["gold_label"] not in name2model:
                    continue
                prem.append(row["sentence1"]); hyp.append(row["sentence2"])
                lab.append(name2model[row["gold_label"]])
        return {"premise": prem, "hypothesis": hyp, "label": np.array(lab)}
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
    lab = np.array(ds["label"])
    keep = lab >= 0
    return {
        "premise": [p for p, k in zip(ds["premise"], keep) if k],
        "hypothesis": [h for h, k in zip(ds["hypothesis"], keep) if k],
        "label": GLUE_TO_MODEL[lab[keep]],
    }


# ----------------------------------------------------------------------------- models

def model_source(i, args):
    """Return (local_dir, revision_sha). Downloads from the Hub unless --local_tmpl is set."""
    if args.local_tmpl:
        return Path(args.local_tmpl.format(i=i)), "local"
    from huggingface_hub import snapshot_download
    repo = REPO_TMPL.format(i=i)
    path = snapshot_download(
        repo,
        allow_patterns=["config.json", "pytorch_model.bin", "*.safetensors", "tokenizer*",
                        "vocab.txt", "special_tokens_map.json"],
    )
    return Path(path), Path(path).name  # snapshot dir name is the commit sha


def purge_from_cache(i):
    from huggingface_hub import scan_cache_dir
    repo = REPO_TMPL.format(i=i)
    info = scan_cache_dir()
    for r in info.repos:
        if r.repo_id == repo:
            info.delete_revisions(*[rev.commit_hash for rev in r.revisions]).execute()


def load_model(path, device):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    tok = AutoTokenizer.from_pretrained(path)
    model = BertForSequenceClassification.from_pretrained(path).to(device).eval()
    # Sanity: uncased tokenizer and 3-way head.
    if tok("Hello World")["input_ids"] != tok("hello world")["input_ids"]:
        print("  WARNING: tokenizer is not lower-casing; McCoy models are bert-base-uncased.")
    assert model.config.num_labels == 3, model.config.num_labels
    return tok, model


def run_model(tok, model, premises, hypotheses, device, batch_size=256):
    """Logits (float32, original order). Batches are formed over length-sorted inputs."""
    import torch
    n = len(premises)
    order = np.argsort([len(p) + len(h) for p, h in zip(premises, hypotheses)])
    out = np.empty((n, 3), dtype=np.float32)
    with torch.inference_mode():
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            enc = tok([premises[j] for j in idx], [hypotheses[j] for j in idx],
                      truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            out[idx] = model(**enc).logits.float().cpu().numpy()
    return out


def pick_device(arg):
    import torch
    if arg != "auto":
        return arg
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def parse_models(spec):
    if spec == ["all"]:
        return list(range(N_MODELS))
    return [int(x) for x in spec]


# ----------------------------------------------------------------------------- step 2: check

def read_stored_eval(path_dir, i, args):
    """The repo's own mnli_eval_bert_{i}.json: label_dict + samples with logits and label."""
    fname = f"mnli_eval_bert_{i}.json"
    if args.local_tmpl:
        p = Path(path_dir) / fname
    else:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(REPO_TMPL.format(i=i), fname)
    with open(p) as f:
        d = json.load(f)
    return d["label_dict"], d["samples"]


def split_stored_sentence(s):
    s = s.strip()
    if s.startswith("[CLS]"):
        s = s[len("[CLS]"):]
    parts = [p.strip() for p in s.split("[SEP]")]
    parts = [p for p in parts if p]
    if len(parts) != 2:
        return None
    return parts[0], parts[1]


def cmd_check(args):
    import torch
    device = pick_device(args.device)
    print(f"device={device}  torch={torch.__version__}")
    mnli = load_mnli(args.mnli_tsv)
    print(f"MNLI matched dev: {len(mnli['label'])} labelled examples")
    rows, all_ok = [], True
    for i in parse_models(args.models):
        t0 = time.time()
        path, sha = model_source(i, args)
        tok, model = load_model(path, device)

        # (a) label order
        label_dict, samples = read_stored_eval(path, i, args)
        label_ok = {k.lower(): int(v) for k, v in label_dict.items()} == MODEL_LABELS

        # (b) reproduce stored logits on the first N stored examples
        pairs, stored = [], []
        for smp in samples[: args.n_reproduce]:
            pr = split_stored_sentence(smp["sentence"])
            if pr is not None:
                pairs.append(pr); stored.append(smp["logits"])
        stored = np.array(stored, dtype=np.float32)
        ours = run_model(tok, model, [p for p, _ in pairs], [h for _, h in pairs], device,
                         args.batch_size)
        absdiff = np.abs(ours - stored)
        argmax_agree = float((ours.argmax(1) == stored.argmax(1)).mean())

        # accuracy implied by the stored file itself (all samples)
        st_logits = np.array([s["logits"] for s in samples], dtype=np.float32)
        st_labels = np.array([s["label"] for s in samples])
        stored_acc = float((st_logits.argmax(1) == st_labels).mean())

        # (c) our accuracy on MNLI matched dev with our label mapping
        logits = run_model(tok, model, mnli["premise"], mnli["hypothesis"], device, args.batch_size)
        acc = float((logits.argmax(1) == mnli["label"]).mean())

        lo, hi = MCCOY_MNLI_RANGE
        ok = (label_ok and argmax_agree >= 0.995 and float(np.median(absdiff)) < 1e-2
              and lo - 0.005 <= acc <= hi + 0.005 and abs(acc - stored_acc) < 0.005)
        all_ok &= ok
        rows.append(dict(model=i, sha=sha, label_order_ok=label_ok, n_stored=len(samples),
                         n_reproduced=len(pairs), median_absdiff=float(np.median(absdiff)),
                         max_absdiff=float(absdiff.max()), argmax_agree=argmax_agree,
                         stored_file_acc=stored_acc, our_mnli_acc=acc,
                         seconds=round(time.time() - t0, 1), PASS=ok))
        r = rows[-1]
        print(f"[model {i:>2}] label_order={'ok' if label_ok else 'MISMATCH'}  "
              f"reproduce: argmax_agree={argmax_agree:.4f} median|dlogit|={r['median_absdiff']:.2e} "
              f"max|dlogit|={r['max_absdiff']:.2e}  stored_acc={stored_acc:.4f}  "
              f"our_acc={acc:.4f}  ({r['seconds']}s)  {'PASS' if ok else 'FAIL'}")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    with open(out / "step2_check.json", "w") as f:
        json.dump({"rows": rows, "env": env_info(), "all_pass": all_ok}, f, indent=2)
    print("\nSTEP 2:", "PASS - population is usable; run `eval`." if all_ok else
          "FAIL - inspect step2_check.json before continuing.")
    print("Interpretation: our_acc ~= stored_file_acc and argmax_agree ~= 1 means weights, "
          "tokenisation and label mapping all match the original evaluation.")


# ----------------------------------------------------------------------------- step 3a: eval

def cmd_eval(args):
    import torch
    device = pick_device(args.device)
    out = Path(args.out_dir); (out / "logits").mkdir(parents=True, exist_ok=True)
    mnli = load_mnli(args.mnli_tsv)
    hans = load_hans(args.hans_path)
    # Save aligned metadata once so `analyze` never depends on loader order at analysis time.
    np.savez(out / "meta.npz", mnli_label=mnli["label"], hans_gold_ent=hans["gold_ent"],
             hans_heuristic=hans["heuristic"], hans_subcase=hans["subcase"],
             hans_template=hans["template"], hans_pairID=hans["pairID"])
    models = parse_models(args.models)
    print(f"device={device}  models={len(models)}  MNLI n={len(mnli['label'])}  HANS n=30000")
    failed = {}
    for k, i in enumerate(models):
        f = out / "logits" / f"model_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            path, sha = model_source(i, args)
            tok, model = load_model(path, device)
            ml = run_model(tok, model, mnli["premise"], mnli["hypothesis"], device, args.batch_size)
            hl = run_model(tok, model, hans["premise"], hans["hypothesis"], device, args.batch_size)
        except Exception as e:  # one bad repo must not kill a 100-model run; rerun later to retry
            failed[i] = f"{type(e).__name__}: {e}"
            print(f"[{k+1}/{len(models)}] model {i}: FAILED ({failed[i][:200]})")
            continue
        tmp = f.with_suffix(".tmp.npz")
        np.savez(tmp, mnli_logits=ml, hans_logits=hl, sha=sha)
        os.replace(tmp, f)  # atomic: a crash never leaves a half-written file
        mnli_acc = (ml.argmax(1) == mnli["label"]).mean()
        lo = (hans["heuristic"] == "lexical_overlap") & ~hans["gold_ent"]
        lo_acc = (hl[lo].argmax(1) != ENT).mean()
        print(f"[{k+1}/{len(models)}] model {i}: MNLI={mnli_acc:.4f}  HANS-LO-nonent={lo_acc:.4f}  "
              f"({time.time()-t0:.0f}s)")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
        if args.purge and not args.local_tmpl:
            purge_from_cache(i)
    with open(out / "eval_env.json", "w") as fh:
        json.dump({**env_info(), "failed": failed}, fh, indent=2)
    done = len(list((out / "logits").glob("model_*.npz")))
    print(f"\nDone: {done} models have logits. Failed this run: {sorted(failed) or 'none'}"
          + ("  (rerun the same command to retry; finished models are skipped)" if failed else ""))


# ----------------------------------------------------------------------------- step 3b: analyze

def bimodality_coefficient(x):
    from scipy.stats import kurtosis, skew
    n = len(x)
    g, k = skew(x, bias=False), kurtosis(x, bias=False)
    return (g ** 2 + 1) / (k + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))


def cmd_analyze(args):
    import itertools
    import pandas as pd
    from scipy.stats import pearsonr, spearmanr

    out = Path(args.out_dir)
    meta = np.load(out / "meta.npz", allow_pickle=True)
    files = sorted((out / "logits").glob("model_*.npz"))
    if len(files) < 10:
        sys.exit(f"Only {len(files)} models evaluated; run `eval` first.")
    ids = [int(f.stem.split("_")[1]) for f in files]
    gold_ent, heur, sub = meta["hans_gold_ent"], meta["hans_heuristic"], meta["hans_subcase"]

    # correctness matrices: models x items
    mnli_correct, hans_correct = [], []
    for f in files:
        z = np.load(f)
        mnli_correct.append(z["mnli_logits"].argmax(1) == meta["mnli_label"])
        pred_ent = z["hans_logits"].argmax(1) == ENT
        hans_correct.append(pred_ent == gold_ent)
    mnli_correct = np.array(mnli_correct); hans_correct = np.array(hans_correct)

    # per-model table
    table = pd.DataFrame({"model": ids, "mnli_acc": mnli_correct.mean(1),
                          "hans_acc": hans_correct.mean(1)})
    for h in ["lexical_overlap", "subsequence", "constituent"]:
        for g, gname in [(True, "ent"), (False, "nonent")]:
            m = (heur == h) & (gold_ent == g)
            table[f"{h}_{gname}"] = hans_correct[:, m].mean(1)
    for s in sorted(set(sub)):
        table[f"sub_{s}"] = hans_correct[:, sub == s].mean(1)
    table.to_csv(out / "per_model_accuracy.csv", index=False)

    # ---- primary outcome: lexical-overlap non-entailment accuracy
    lo_mask = (heur == "lexical_overlap") & ~gold_ent
    Y = hans_correct[:, lo_mask]            # models x 5000
    lo_sub = sub[lo_mask]
    y = Y.mean(1)
    n_items = Y.shape[1]
    rng = np.random.default_rng(args.seed)

    # (1) spread vs. binomial sampling noise
    p = y.mean()
    samp_sd = np.sqrt(p * (1 - p) / n_items)
    between_sd = y.std(ddof=1)
    rng_ = y.max() - y.min()

    # (2) stratified random split-half reliability (each subcase split in half)
    rs = []
    sub_idx = {s: np.where(lo_sub == s)[0] for s in np.unique(lo_sub)}
    for _ in range(args.n_splits):
        a = np.zeros(n_items, bool)
        for s, ix in sub_idx.items():
            perm = rng.permutation(ix)
            a[perm[: len(ix) // 2]] = True
        rs.append(pearsonr(Y[:, a].mean(1), Y[:, ~a].mean(1))[0])
    rs = np.array(rs)
    r_half = float(np.mean(rs))
    r_full = 2 * r_half / (1 + r_half)    # Spearman-Brown: reliability of the full 5000-item score

    # (3) stricter: subcase split (do constructions agree about which models generalise?)
    subs = sorted(sub_idx)
    sub_acc = np.stack([Y[:, sub_idx[s]].mean(1) for s in subs], 1)
    sub_corr = pd.DataFrame(np.corrcoef(sub_acc.T), index=subs, columns=subs)
    part_r = []
    for k in (2,):
        for A in itertools.combinations(range(len(subs)), k):
            B = [j for j in range(len(subs)) if j not in A]
            part_r.append(pearsonr(sub_acc[:, list(A)].mean(1), sub_acc[:, B].mean(1))[0])

    # (4) diagnostics for later steps
    mnli_acc = mnli_correct.mean(1)
    r_mnli = pearsonr(mnli_acc, y)[0]
    rho_mnli = spearmanr(mnli_acc, y)[0]
    bc = bimodality_coefficient(y)

    g_rel = r_half >= GATE_SPLIT_HALF_R
    g_spread = (between_sd >= GATE_SD_RATIO * samp_sd) and (rng_ >= GATE_RANGE)
    lo_, hi_ = MCCOY_MNLI_RANGE

    lines = [
        f"STEP 3 ANALYSIS  ({len(ids)} models)",
        "",
        f"MNLI matched-dev accuracy: min {mnli_acc.min():.4f}  max {mnli_acc.max():.4f}  "
        f"(McCoy et al. report {lo_:.3f}-{hi_:.3f})",
        "",
        f"Primary outcome: HANS lexical-overlap non-entailment accuracy (n={n_items} items/model)",
        f"  mean {y.mean():.4f}  SD {between_sd:.4f}  min {y.min():.4f}  max {y.max():.4f}  "
        f"IQR {np.percentile(y,25):.4f}-{np.percentile(y,75):.4f}",
        f"  binomial sampling SD at this mean: {samp_sd:.4f}  ->  between-model/sampling SD ratio "
        f"{between_sd/samp_sd:.1f}",
        "",
        f"Reliability (stratified random split-half, {args.n_splits} splits):",
        f"  raw split-half r  mean {r_half:.3f}  [2.5%-97.5%: {np.percentile(rs,2.5):.3f}-"
        f"{np.percentile(rs,97.5):.3f}]",
        f"  Spearman-Brown full-length reliability {r_full:.3f}",
        f"  -> ceiling for any predictor: R^2 <= {r_full:.3f}, |r| <= {np.sqrt(r_full):.3f}",
        "",
        "Stricter: agreement across constructions (2-vs-3 subcase partitions, r):",
        "  " + "  ".join(f"{v:.2f}" for v in part_r) + f"   (mean {np.mean(part_r):.3f})",
        "  Between-subcase correlation matrix:",
        sub_corr.round(2).to_string(),
        "",
        "Diagnostics for step 4:",
        f"  MNLI accuracy vs outcome: Pearson {r_mnli:.3f}, Spearman {rho_mnli:.3f}  "
        f"(R^2 = {r_mnli**2:.3f}; share of reliable variance = {r_mnli**2 / r_full:.2f})",
        f"  bimodality coefficient {bc:.3f}  (>0.555 suggests two clusters, cf. Juneja's two basins)",
        "",
        "GATES (pre-registered):",
        f"  G1 reliability: raw split-half r >= {GATE_SPLIT_HALF_R}  ->  {'PASS' if g_rel else 'FAIL'}",
        f"  G2 spread: SD ratio >= {GATE_SD_RATIO} and range >= {GATE_RANGE}  ->  "
        f"{'PASS' if g_spread else 'FAIL'}",
        "",
        ("VERDICT: outcome is real; proceed to step 4 (baselines)." if (g_rel and g_spread) else
         "VERDICT: outcome fails a gate; OOD differences are mostly noise or too small. Stop or "
         "change the outcome before building predictors."),
    ]
    if r_mnli ** 2 / r_full > 0.5:
        lines.append("NOTE: MNLI accuracy alone already explains >50% of the reliable variance; "
                     "step 4 headroom may be small.")
    report = "\n".join(lines)
    print(report)
    (out / "step3_report.txt").write_text(report + "\n")
    with open(out / "step3_summary.json", "w") as f:
        json.dump(dict(n_models=len(ids), outcome_mean=float(y.mean()), outcome_sd=float(between_sd),
                       outcome_min=float(y.min()), outcome_max=float(y.max()),
                       sampling_sd=float(samp_sd), split_half_r=r_half, spearman_brown=r_full,
                       subcase_partition_r=[float(v) for v in part_r], mnli_pearson=float(r_mnli),
                       bimodality_coef=float(bc), gate_reliability=bool(g_rel),
                       gate_spread=bool(g_spread)), f, indent=2)


# ----------------------------------------------------------------------------- misc

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform()}
    for mod in ["torch", "transformers", "numpy", "scipy", "huggingface_hub", "datasets"]:
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["cuda"] = torch.version.cuda
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        pass
    return info


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["check", "eval", "analyze"]:
        p = sp.add_parser(name)
        p.add_argument("--out_dir", default="feather_out")
        p.add_argument("--seed", type=int, default=0)
        if name in ("check", "eval"):
            p.add_argument("--models", nargs="+", default=["0", "1", "2", "3", "4"] if name == "check" else ["all"])
            p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=256)
            p.add_argument("--mnli_tsv", default=None, help="GLUE dev_matched.tsv (else HF datasets)")
            p.add_argument("--local_tmpl", default=None,
                           help="Local model dirs, e.g. /models/feather_berts_{i} (skips the Hub)")
        if name == "check":
            p.add_argument("--n_reproduce", type=int, default=2000,
                           help="stored examples to re-run for the logit comparison")
        if name == "eval":
            p.add_argument("--hans_path", default="data/heuristics_evaluation_set.txt")
            p.add_argument("--purge", action="store_true",
                           help="delete each model from the HF cache after use (saves ~44 GB)")
        if name == "analyze":
            p.add_argument("--n_splits", type=int, default=1000)
    args = ap.parse_args()
    {"check": cmd_check, "eval": cmd_eval, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile feather_step4.py
#!/usr/bin/env python3
"""
Step 4: do internal (mechanistic) signals predict HANS lexical-overlap generalisation better than
in-distribution behaviour, weight-space and representational baselines?

Needs feather_eval.py (steps 2-3) in the same folder and its output folder (feather_out/ with
meta.npz and logits/).

Subcommands
-----------
  split    Freeze a 70/30 split of the 100 models (seed fixed). Do this ONCE, before anything else.
           The 30 test models are not looked at until `analyze --final`.
  extract  GPU. For every model, compute features from MNLI matched-dev inputs ONLY (never HANS):
             - representations (mean-pooled hidden states at 4 depths) on a 512-item probe set
             - attention statistics: attention from hypothesis tokens to identical premise tokens
             - head-ablation profile: mean-ablate each head, change in entailment log-odds on
               high-overlap and on ordinary probe items
             - weights: per-block distance from pretrained BERT, a 4096-d CountSketch of the
               fine-tuning delta, and the classifier weights
           Resumable; --purge deletes each model from the HF cache afterwards.
  analyze  CPU. Kernel ridge regression from each feature block to the outcome (several blocks: stacked
           with non-negative weights fitted on leave-one-out predictions). Development
           models: repeated 10-fold cross-validation. With --final: fit on all 70 development
           models and score once on the 30 test models.

Outcome: HANS lexical-overlap non-entailment accuracy (as in step 3). Secondary: same without the
ln_passive subcase.

Pre-registered decision (primary contrast, on the 30 test models):
  dR2 = R2(baselines + internal) - R2(baselines) >= 0.10 and its 95% bootstrap CI excludes 0
  -> GO.  Internal signals predict but dR2 < 0.10 or CI includes 0 -> PIVOT.  Nothing beats the
  MNLI-accuracy-only model -> STOP.
"""

import argparse
import json
import re
import sys
import time
import zlib
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import feather_eval as fe  # noqa: E402

N_PROBE_HO = 256        # high-overlap probe items
N_PROBE_RAND = 256      # ordinary probe items
HO_THRESH = 0.8         # share of hypothesis words that also occur in the premise
SKETCH_D = 4096
SPLIT_SEED = 2026
N_TEST = 30
PRETRAINED = "google-bert/bert-base-uncased"
MIN_DELTA_R2 = 0.10     # pre-registered GO threshold


# ----------------------------------------------------------------------------- helpers

def word_overlap(p, h):
    hw = re.findall(r"\w+", h.lower())
    pw = set(re.findall(r"\w+", p.lower()))
    return float(np.mean([w in pw for w in hw])) if hw else 0.0


def ent_logodds(logits):
    """log P(entailment) - log P(non-entailment), from 3-way logits (numpy or torch)."""
    import torch
    t = torch.as_tensor(logits)
    others = [k for k in range(3) if k != fe.ENT]
    return (t[..., fe.ENT] - torch.logsumexp(t[..., others], -1))


# ----------------------------------------------------------------------------- split

def cmd_split(args):
    out = Path(args.out_dir)
    f = out / "split.json"
    if f.exists() and not args.force:
        print(f"{f} already exists; refusing to overwrite (that would leak the test set).")
        print(json.loads(f.read_text()))
        return
    perm = np.random.default_rng(SPLIT_SEED).permutation(fe.N_MODELS)
    split = {"seed": SPLIT_SEED, "test": sorted(int(i) for i in perm[:N_TEST]),
             "dev": sorted(int(i) for i in perm[N_TEST:]),
             "created": time.strftime("%Y-%m-%d %H:%M:%S")}
    out.mkdir(parents=True, exist_ok=True)
    f.write_text(json.dumps(split, indent=1))
    print(f"Wrote {f}: {len(split['dev'])} dev, {len(split['test'])} test models.")


# ----------------------------------------------------------------------------- probe set

def build_probe(mnli, out):
    """Fixed probe set from MNLI matched dev: high-overlap items + ordinary items."""
    fdir = out / "features"; fdir.mkdir(parents=True, exist_ok=True)
    pf = fdir / "probe.json"
    ov = np.array([word_overlap(p, h) for p, h in zip(mnli["premise"], mnli["hypothesis"])])
    np.save(fdir / "mnli_overlap.npy", ov)
    if pf.exists():
        probe = json.loads(pf.read_text())
    else:
        rng = np.random.default_rng(0)
        ho = np.where(ov >= HO_THRESH)[0]; rest = np.where(ov < HO_THRESH)[0]
        ho_sel = np.sort(rng.choice(ho, min(N_PROBE_HO, len(ho)), replace=False))
        rd_sel = np.sort(rng.choice(rest, min(N_PROBE_RAND, len(rest)), replace=False))
        probe = {"idx": [int(i) for i in np.concatenate([ho_sel, rd_sel])],
                 "is_ho": [True] * len(ho_sel) + [False] * len(rd_sel),
                 "n_ho_in_dev": int(len(ho)),
                 "n_ho_nonent_in_dev": int(((ov >= HO_THRESH) & (mnli["label"] != fe.ENT)).sum())}
        pf.write_text(json.dumps(probe))
    print(f"MNLI dev: {probe['n_ho_in_dev']} high-overlap items "
          f"({probe['n_ho_nonent_in_dev']} non-entailment); probe set = {len(probe['idx'])} items")
    return probe


# ----------------------------------------------------------------------------- extract

def get_layers(model):
    return model.bert.encoder.layer


def encode_batches(tok, prem, hyp, device, bs):
    order = np.argsort([len(p) + len(h) for p, h in zip(prem, hyp)])
    batches = []
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True,
                  max_length=fe.MAX_LEN, padding=True, return_tensors="pt")
        batches.append((idx, {k: v.to(device) for k, v in enc.items()}))
    return batches


def extract_one(i, args, device, probe_text, is_ho, pre_sd):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    path, sha = fe.model_source(i, args)
    tok = AutoTokenizer.from_pretrained(path)
    try:
        model = BertForSequenceClassification.from_pretrained(path, attn_implementation="eager")
    except (TypeError, ValueError):
        model = BertForSequenceClassification.from_pretrained(path)
    model = model.to(device).eval()
    cfg = model.config
    L, NH, H = cfg.num_hidden_layers, cfg.num_attention_heads, cfg.hidden_size
    dh = H // NH
    layers = get_layers(model)
    rep_layers = [L // 4, L // 2, (3 * L) // 4, L]
    N = len(probe_text[0])
    batches = encode_batches(tok, probe_text[0], probe_text[1], device, args.batch_size)
    is_ho_t = torch.tensor(is_ho, device=device)
    cls_id, sep_id = tok.cls_token_id, tok.sep_token_id

    # ---- clean pass: logits, representations, attention stats, per-layer context means
    ctx_sum = torch.zeros(L, H, device=device, dtype=torch.float64); ntok = 0
    hooks, cur_mask = [], {}

    def make_capture(l):
        def hook(mod, inp):
            x = inp[0]; m = cur_mask["m"].unsqueeze(-1).to(x.dtype)
            ctx_sum[l] += (x * m).sum((0, 1)).double()
        return hook

    for l in range(L):
        hooks.append(layers[l].attention.output.dense.register_forward_pre_hook(make_capture(l)))
    clean_lo = torch.empty(N, device=device)
    rep = torch.empty(len(rep_layers), N, H, device=device)
    match_sum = torch.zeros(L, NH, device=device, dtype=torch.float64); match_n = 0
    prem_sum = torch.zeros(L, NH, device=device, dtype=torch.float64); prem_n = 0
    with torch.inference_mode():
        for idx, enc in batches:
            am = enc["attention_mask"]; cur_mask["m"] = am; ntok += int(am.sum())
            o = model(**enc, output_hidden_states=True, output_attentions=True)
            it = torch.as_tensor(idx, device=device)
            clean_lo[it] = ent_logodds(o.logits.float())
            amf = am.unsqueeze(-1).float()
            for r, l in enumerate(rep_layers):
                hs = o.hidden_states[l].float()
                rep[r, it] = (hs * amf).sum(1) / amf.sum(1)
            # attention statistics on high-overlap items only
            ids, tt = enc["input_ids"], enc["token_type_ids"]
            special = (ids == cls_id) | (ids == sep_id) | (am == 0)
            if tok.unk_token_id is not None:
                special = special | (ids == tok.unk_token_id)
            hypq = (tt == 1) & ~special
            premk = (tt == 0) & ~special
            match = (ids.unsqueeze(2) == ids.unsqueeze(1)) & hypq.unsqueeze(2) & premk.unsqueeze(1)
            sel = is_ho_t[it]
            q_has = match.any(-1) & sel.unsqueeze(1)                 # (B,T) queries with a match
            q_hyp = hypq & sel.unsqueeze(1)
            att = torch.stack([a.float() for a in o.attentions], 1)  # (B,L,NH,T,T)
            mass_match = (att * match[:, None, None].float()).sum(-1)  # (B,L,NH,T)
            mass_prem = (att * premk[:, None, None, None, :].float()).sum(-1)
            match_sum += (mass_match * q_has[:, None, None].float()).sum((0, 3)).double()
            match_n += int(q_has.sum())
            prem_sum += (mass_prem * q_hyp[:, None, None].float()).sum((0, 3)).double()
            prem_n += int(q_hyp.sum())
            del o, att
    for hk in hooks:
        hk.remove()
    ctx_mean = (ctx_sum / ntok).float()
    attn_match = (match_sum / max(match_n, 1)).float().cpu().numpy()
    attn_prem = (prem_sum / max(prem_n, 1)).float().cpu().numpy()

    # ---- head-ablation profile (mean ablation of each head's context vector)
    abl_ho = np.zeros((L, NH), np.float32); abl_rd = np.zeros((L, NH), np.float32)
    ho_np = np.array(is_ho)
    base = clean_lo.cpu().numpy()
    use_amp = device == "cuda" and not args.fp32_ablation

    def run_lo():
        lo = np.empty(N, np.float32)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            for idx, enc in batches:
                lo[idx] = ent_logodds(model(**enc).logits.float()).cpu().numpy()
        return lo

    with torch.inference_mode():
        base_abl = run_lo()  # clean reference at the same precision as the ablated runs
        amp_err = float(np.abs(base_abl - base).max())
        for l in range(L):
            for h in range(NH):
                sl = slice(h * dh, (h + 1) * dh)

                def hook(mod, inp, sl=sl, l=l):
                    x = inp[0].clone(); x[..., sl] = ctx_mean[l, sl].to(x.dtype)
                    return (x,)
                hk = layers[l].attention.output.dense.register_forward_pre_hook(hook)
                lo = run_lo()
                hk.remove()
                d = lo - base_abl
                abl_ho[l, h] = d[ho_np].mean() if ho_np.any() else 0.0
                abl_rd[l, h] = d[~ho_np].mean()

    # ---- weights: distance from pretrained, CountSketch of the delta, classifier
    ft = model.bert.state_dict()
    groups = ["embeddings"] + [f"layer.{k}" for k in range(L)] + ["pooler"]
    gsq = dict.fromkeys(groups, 0.0); gsq0 = dict.fromkeys(groups, 0.0)
    sketch = torch.zeros(SKETCH_D, device=device, dtype=torch.float64)
    with torch.inference_mode():
        for key in sorted(ft):
            w = ft[key]
            if key not in pre_sd or not torch.is_floating_point(w) or w.shape != pre_sd[key].shape:
                continue
            w0 = pre_sd[key].to(device=device, dtype=torch.float32)
            d = (w.float() - w0).flatten()
            g = next((gr for gr in groups if f"{gr}." in key or key.startswith(gr)), None)
            if g is not None:
                gsq[g] += float((d.double() ** 2).sum()); gsq0[g] += float((w0.double() ** 2).sum())
            gen = torch.Generator(device=device).manual_seed(zlib.crc32(key.encode()))
            b = torch.randint(0, SKETCH_D, (d.numel(),), generator=gen, device=device)
            s = torch.randint(0, 2, (d.numel(),), generator=gen, device=device).float() * 2 - 1
            sketch.index_add_(0, b, (d * s).double())
    dnorm = np.array([np.sqrt(gsq[g]) for g in groups], np.float32)
    drel = np.array([np.sqrt(gsq[g] / gsq0[g]) if gsq0[g] > 0 else 0 for g in groups], np.float32)
    clf = torch.cat([model.classifier.weight.flatten(), model.classifier.bias]).detach().float().cpu().numpy()

    res = dict(rep=rep.cpu().numpy().astype(np.float16), rep_layers=np.array(rep_layers),
               attn_match=attn_match, attn_prem=attn_prem, abl_ho=abl_ho, abl_rand=abl_rd,
               clean_lo=base, delta_norm=dnorm, delta_rel=drel, groups=np.array(groups),
               sketch=sketch.float().cpu().numpy(), clf=clf, sha=sha, device=device,
               amp_max_err=amp_err)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return res


def cmd_extract(args):
    import torch
    from transformers import BertModel
    device = fe.pick_device(args.device)
    out = Path(args.out_dir); fdir = out / "features"; fdir.mkdir(parents=True, exist_ok=True)
    meta = np.load(out / "meta.npz", allow_pickle=True)
    mnli = fe.load_mnli(args.mnli_tsv)
    assert np.array_equal(mnli["label"], meta["mnli_label"]), \
        "MNLI order/labels differ from step 3 - cannot align with saved logits"
    probe = build_probe(mnli, out)
    pidx = probe["idx"]
    probe_text = ([mnli["premise"][j] for j in pidx], [mnli["hypothesis"][j] for j in pidx])
    print(f"Loading pretrained reference {args.pretrained} ...")
    pre_sd = {k: v.detach().cpu() for k, v in BertModel.from_pretrained(args.pretrained).state_dict().items()}
    models = fe.parse_models(args.models)
    print(f"device={device} models={len(models)}")
    failed = {}
    for k, i in enumerate(models):
        f = fdir / f"feat_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            res = extract_one(i, args, device, probe_text, probe["is_ho"], pre_sd)
        except Exception as e:
            failed[i] = f"{type(e).__name__}: {e}"
            print(f"[{k+1}/{len(models)}] model {i}: FAILED ({failed[i][:300]})")
            if device == "cuda":
                torch.cuda.empty_cache()
            continue
        tmp = f.with_suffix(".tmp.npz"); np.savez(tmp, **res); tmp.replace(f)
        print(f"[{k+1}/{len(models)}] model {i}: ok  max|abl_ho|={np.abs(res['abl_ho']).max():.3f}  "
              f"fp16 err={res['amp_max_err']:.1e}  "
              f"({time.time()-t0:.0f}s)")
        if args.purge and not args.local_tmpl:
            fe.purge_from_cache(i)
    (out / "step4_extract_env.json").write_text(json.dumps({**fe.env_info(), "failed": failed}, indent=2))
    done = len(list(fdir.glob("feat_*.npz")))
    print(f"\nDone: {done} models have features. Failed this run: {sorted(failed) or 'none'}")


# ----------------------------------------------------------------------------- analyze

def center_kernels(Ktr, Kte):
    n = Ktr.shape[0]
    mr = Ktr.mean(0); ma = Ktr.mean()
    Ktr_c = Ktr - mr[None] - mr[:, None] + ma
    Kte_c = Kte - mr[None] - Kte.mean(1, keepdims=True) + ma
    return Ktr_c, Kte_c


ALPHAS = np.logspace(-4, 3, 22)


def krr(Ktr, Kte, ytr):
    """Kernel ridge with intercept; alpha by exact leave-one-out on the training models."""
    Ktr_c, Kte_c = center_kernels(Ktr, Kte)
    yc = ytr - ytr.mean()
    w, V = np.linalg.eigh(Ktr_c)
    w = np.clip(w, 0, None)
    best = (np.inf, None, None)
    Vy = V.T @ yc
    for a in ALPHAS:
        Hdiag = (V ** 2) @ (w / (w + a)) + 1.0 / len(ytr)  # + the intercept's share of the hat matrix
        fit = V @ (w / (w + a) * Vy)
        res = (yc - fit) / np.clip(1 - Hdiag, 1e-6, None)   # exact leave-one-out residuals
        if (res ** 2).mean() < best[0]:
            best = ((res ** 2).mean(), a, ytr - res)
    a = best[1]
    coef = V @ (Vy / (w + a))
    return ytr.mean() + Kte_c @ coef, a, best[2]


def r2(y, p):
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()


def linear_kernel_block(X):
    """Returns f(tr, rows) -> kernel using train-fold standardisation."""
    X = np.asarray(X, np.float64)

    def f(tr):
        mu = X[tr].mean(0); sd = X[tr].std(0); keep = sd > 1e-12
        Z = (X[:, keep] - mu[keep]) / sd[keep]
        K = Z @ Z.T / max(keep.sum(), 1)
        return K / np.mean(np.diag(K)[tr])
    return f


def rbf_distance_block(X):
    """Juneja-style weight geometry: Gaussian kernel on Euclidean distances between models'
    weights (the CountSketch of the fine-tuning delta preserves these distances). Juneja et al.
    report Euclidean distance behaves similarly to their convexity-gap metric. Bandwidth =
    median distance among the training models only."""
    X = np.asarray(X, np.float64)
    D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)

    def f(tr):
        d = D2[np.ix_(tr, tr)][np.triu_indices(len(tr), 1)]
        return np.exp(-D2 / np.median(d))
    return f


def fixed_kernel_block(K):
    def f(tr):
        return K / np.mean(np.diag(K)[tr])
    return f


def linear_cka_kernel(reps):
    """reps: models x N x H -> models x models linear CKA."""
    F = []
    for X in reps:
        X = X.astype(np.float64); X = X - X.mean(0)
        G = X @ X.T
        F.append((G / np.linalg.norm(G)).ravel())
    F = np.array(F)
    return F @ F.T


def predict_set(blocks, names, tr, te, y):
    """One block: kernel ridge. Several blocks: stacking - a kernel ridge per block, then
    non-negative weights fitted to the blocks' leave-one-out predictions on the training models,
    so one strong block is not diluted by weak ones."""
    if len(names) == 1:
        K = blocks[names[0]](tr)
        return krr(K[np.ix_(tr, tr)], K[np.ix_(te, tr)], y[tr])[0]
    from scipy.optimize import nnls
    P_loo, P_te = [], []
    for n in names:
        K = blocks[n](tr)
        pte, _, ploo = krr(K[np.ix_(tr, tr)], K[np.ix_(te, tr)], y[tr])
        P_loo.append(ploo); P_te.append(pte)
    P_loo = np.array(P_loo).T; P_te = np.array(P_te).T
    mu = P_loo.mean(0); ym = y[tr].mean()
    wts, _ = nnls(P_loo - mu, y[tr] - ym)
    return ym + (P_te - mu) @ wts


def cv_oof(blocks, names, idx, y, n_rep, seed):
    """Repeated 10-fold CV on the model indices `idx`; returns mean OOF prediction and R2s."""
    rng = np.random.default_rng(seed)
    preds = np.zeros((n_rep, len(idx)))
    for r in range(n_rep):
        folds = np.array_split(rng.permutation(len(idx)), 10)
        for fo in folds:
            te = idx[fo]; tr = idx[np.setdiff1d(np.arange(len(idx)), fo)]
            preds[r, fo] = predict_set(blocks, names, tr, te, y)
    r2s = np.array([r2(y[idx], p) for p in preds])
    return preds.mean(0), r2s


def boot_delta(y, pa, pb, n=4000, seed=0):
    rng = np.random.default_rng(seed); m = len(y); d = []
    for _ in range(n):
        b = rng.integers(0, m, m)
        if np.var(y[b]) == 0:
            continue
        d.append(r2(y[b], pa[b]) - r2(y[b], pb[b]))
    return np.percentile(d, [2.5, 97.5])


def build_blocks(out, ids):
    meta = np.load(out / "meta.npz", allow_pickle=True)
    fdir = out / "features"
    ov = np.load(fdir / "mnli_overlap.npy")
    lab = meta["mnli_label"]
    gold, heur, sub = meta["hans_gold_ent"], meta["hans_heuristic"], meta["hans_subcase"]
    lo_mask = (heur == "lexical_overlap") & ~gold
    lo_np = lo_mask & (sub != "ln_passive")
    ho = ov >= HO_THRESH
    y, y_np, idsum, iditems, wts, attn, abl, reps, mnli_acc = [], [], [], [], [], [], [], [], []
    sketch = []
    for i in ids:
        z = np.load(out / "logits" / f"model_{i:03d}.npz")
        hl = z["hans_logits"]; ml = z["mnli_logits"]
        y.append((hl[lo_mask].argmax(1) != fe.ENT).mean())
        y_np.append((hl[lo_np].argmax(1) != fe.ENT).mean())
        pr = np.exp(ml - ml.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
        pred = ml.argmax(1); corr = pred == lab
        rec = [corr[lab == c].mean() for c in range(3)]
        ent = -(pr * np.log(pr + 1e-12)).sum(1)
        hn = ho & (lab != fe.ENT); he = ho & (lab == fe.ENT)
        mnli_acc.append(corr.mean())
        idsum.append([corr.mean(), *rec, pr.max(1).mean(), ent.mean(), pr[:, fe.ENT].mean(),
                      corr[ho].mean(), corr[hn].mean(), pr[hn, fe.ENT].mean(), pr[he, fe.ENT].mean()])
        iditems.append(np.log(pr[ho, fe.ENT] + 1e-9) - np.log(1 - pr[ho, fe.ENT] + 1e-9))
        f = np.load(fdir / f"feat_{i:03d}.npz")
        wts.append(np.concatenate([f["delta_norm"], f["delta_rel"], f["sketch"], f["clf"]]))
        sketch.append(f["sketch"])
        attn.append(np.concatenate([f["attn_match"].ravel(), f["attn_prem"].ravel()]))
        abl.append(np.concatenate([f["abl_ho"].ravel(), f["abl_rand"].ravel()]))
        reps.append(f["rep"])
    reps = np.array(reps)  # models x layers x N x H
    cka = np.mean([linear_cka_kernel(reps[:, l]) for l in range(reps.shape[1])], 0)
    blocks = {
        "MNLI-acc": linear_kernel_block(np.array(mnli_acc)[:, None]),
        "ID-summary": linear_kernel_block(idsum),
        "ID-overlap-items": linear_kernel_block(iditems),
        "Weights": linear_kernel_block(wts),
        "Repr-CKA": fixed_kernel_block(cka),
        "Weight-geometry": rbf_distance_block(sketch),
        "Attn-stats": linear_kernel_block(attn),
        "Head-ablation": linear_kernel_block(abl),
    }
    extra = dict(n_ho=int(ho.sum()), n_ho_nonent=int((ho & (lab != fe.ENT)).sum()))
    return blocks, np.array(y), np.array(y_np), extra


BASE = ["ID-summary", "ID-overlap-items", "Weights", "Weight-geometry", "Repr-CKA"]
INTERNAL = ["Attn-stats", "Head-ablation"]
SETS = {
    "MNLI-acc only": ["MNLI-acc"],
    "ID-summary": ["ID-summary"],
    "ID-overlap-items": ["ID-overlap-items"],
    "Weights": ["Weights"],
    "Weight-geometry (Juneja-style)": ["Weight-geometry"],
    "Repr-CKA": ["Repr-CKA"],
    "Attn-stats": ["Attn-stats"],
    "Head-ablation": ["Head-ablation"],
    "INTERNAL (attn+ablation)": INTERNAL,
    "ALL BASELINES": BASE,
    "ALL BASELINES + INTERNAL": BASE + INTERNAL,
}


def cmd_analyze(args):
    out = Path(args.out_dir)
    split = json.loads((out / "split.json").read_text())
    have = {int(p.stem.split("_")[1]) for p in (out / "features").glob("feat_*.npz")}
    dev = [i for i in split["dev"] if i in have]; test = [i for i in split["test"] if i in have]
    miss = sorted(set(split["dev"] + split["test"]) - have)
    ids = dev + test
    blocks, y_all, ynp_all, extra = build_blocks(out, ids)
    dev_idx = np.arange(len(dev)); test_idx = np.arange(len(dev), len(ids))
    lines = [f"STEP 4 ANALYSIS  ({'FINAL: test models' if args.final else 'development models, CV'})",
             f"models: {len(dev)} dev, {len(test)} test" + (f"; MISSING features: {miss}" if miss else ""),
             f"MNLI high-overlap items (overlap >= {HO_THRESH}): {extra['n_ho']} "
             f"({extra['n_ho_nonent']} non-entailment)", ""]
    summary = {}
    for oname, y in [("HANS-LO non-entailment (primary)", y_all),
                     ("HANS-LO non-entailment without ln_passive (secondary)", ynp_all)]:
        lines.append(f"== Outcome: {oname}")
        preds = {}
        if not args.final:
            lines.append(f"{'feature set':32s} {'CV R2 mean':>10s} {'[min, max over repeats]':>24s}")
            for sname, names in SETS.items():
                p, r2s = cv_oof(blocks, names, dev_idx, y, args.n_rep, args.seed)
                preds[sname] = p
                lines.append(f"{sname:32s} {r2s.mean():10.3f}   [{r2s.min():.3f}, {r2s.max():.3f}]")
            yy = y[dev_idx]
        else:
            lines.append(f"{'feature set':32s} {'test R2':>10s}")
            for sname, names in SETS.items():
                preds[sname] = predict_set(blocks, names, dev_idx, test_idx, y)
                lines.append(f"{sname:32s} {r2(y[test_idx], preds[sname]):10.3f}")
            yy = y[test_idx]
        A, B = "ALL BASELINES + INTERNAL", "ALL BASELINES"
        d = r2(yy, preds[A]) - r2(yy, preds[B]); ci = boot_delta(yy, preds[A], preds[B])
        d2 = r2(yy, preds["INTERNAL (attn+ablation)"]) - r2(yy, preds["ID-overlap-items"])
        ci2 = boot_delta(yy, preds["INTERNAL (attn+ablation)"], preds["ID-overlap-items"])
        lines += ["",
                  f"PRIMARY CONTRAST  dR2 = R2(baselines+internal) - R2(baselines) = {d:+.3f}  "
                  f"95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]",
                  f"secondary         R2(internal) - R2(ID-overlap-items)          = {d2:+.3f}  "
                  f"95% CI [{ci2[0]:+.3f}, {ci2[1]:+.3f}]"]
        r2_int = r2(yy, preds["INTERNAL (attn+ablation)"])
        r2_acc = r2(yy, preds["MNLI-acc only"])
        if d >= MIN_DELTA_R2 and ci[0] > 0:
            verdict = "GO - internal signals add predictive information beyond all baselines."
        elif r2_int >= max(0.05, r2_acc + 0.05):
            verdict = ("PIVOT - internal signals predict the outcome but do not clear the "
                       f"pre-registered bar over baselines (need dR2 >= {MIN_DELTA_R2} with CI > 0).")
        else:
            verdict = ("STOP - internal signals predict no better than MNLI accuracy alone "
                       "(check whether a baseline does: that would be a different paper).")
        lines += [f"VERDICT ({'binding' if args.final else 'provisional, dev CV only'}): {verdict}", ""]
        summary[oname] = dict(delta_r2=float(d), ci=[float(c) for c in ci],
                              r2={k: float(r2(yy, p)) for k, p in preds.items()}, verdict=verdict)
    if args.final:
        lk = out / "step4_test_unlocked.json"
        prev = json.loads(lk.read_text()) if lk.exists() else []
        if prev:
            lines.insert(1, f"WARNING: test set already scored {len(prev)} time(s) before; "
                            "only the first look is confirmatory.")
        prev.append(time.strftime("%Y-%m-%d %H:%M:%S")); lk.write_text(json.dumps(prev))
    rep = "\n".join(lines); print(rep)
    tag = "final" if args.final else "dev"
    (out / f"step4_report_{tag}.txt").write_text(rep + "\n")
    (out / f"step4_summary_{tag}.json").write_text(json.dumps(summary, indent=2))


# ----------------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["split", "extract", "analyze"]:
        p = sp.add_parser(name)
        p.add_argument("--out_dir", default="feather_out")
        p.add_argument("--seed", type=int, default=0)
        if name == "split":
            p.add_argument("--force", action="store_true")
        if name == "extract":
            p.add_argument("--models", nargs="+", default=["all"])
            p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=64)
            p.add_argument("--mnli_tsv", default=None)
            p.add_argument("--local_tmpl", default=None)
            p.add_argument("--pretrained", default=PRETRAINED)
            p.add_argument("--purge", action="store_true")
            p.add_argument("--fp32_ablation", action="store_true",
                           help="run the 144 ablation passes in fp32 (slower; default fp16 on GPU)")
        if name == "analyze":
            p.add_argument("--final", action="store_true",
                           help="score on the 30 held-out test models (do this once)")
            p.add_argument("--n_rep", type=int, default=20)
    args = ap.parse_args()
    {"split": cmd_split, "extract": cmd_extract, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fix1_qqp.py
#!/usr/bin/env python3
"""
Fix 1, stage A: is there a second population with a REAL out-of-distribution outcome?

Population: the 100 BERT-base models Juneja et al. (2023) fine-tuned on QQP (paraphrase detection),
hosted as connectivity/bert_ft_qqp-0 ... -99. Label order (from each repo's eval files):
0 = not_duplicate, 1 = duplicate. The shortcut analogous to HANS lexical overlap: "the two
questions share their words, so they are duplicates". PAWS is built from exactly such traps.

This stage only runs the models and checks the outcome, as step 3 did for HANS:
  run      GPU. For each model: QQP validation (fixed 10k subsample), PAWS-QQP (the 677 dev+test
           items stored in each repo's paws_eval json, sentences included) and PAWS-Wiki
           (labeled_final test, 8,000 items). Saves logits; resumable; --purge frees disk.
  gate     CPU. Reproduction check against the stored PAWS-QQP logits, and the reliability gate
           for each candidate outcome.

  groups   CPU + internet. Writes item groups for the split-half (items sharing a sentence are
           one group), so reliability is not inflated by near-duplicate items.

Pre-stated (written before running; revised 2026-11-14 BEFORE any stage-A result was seen):
  Two DISTINCT evaluation settings, never pooled and never substituted for one another:
    PAWS-QQP   adversarial pairs built from Quora questions (same source domain as training);
    PAWS-Wiki  adversarial pairs built from Wikipedia (adversarial AND a domain shift).
  Outcome in each: accuracy on NON-paraphrase (label 0) items.
  Gate per setting, thresholds as in step 3 (fixed before step 3 was run):
    - group-blocked split-half r >= 0.80 (items sharing a sentence never straddle the halves).
      Rationale: the Spearman-Brown full-length reliability is then >= 0.89, so at most ~11% of
      between-model variance is measurement noise and an increment of dR2 = 0.10 is resolvable;
      0.80 is the conventional floor for score reliability.
    - between-model SD >= 3x the binomial sampling SD (sampling variance <= 1/9 of the observed
      variance, consistent with the reliability floor).
    - range >= 0.10 (smaller differences between models are too small to matter in practice).
  Each passing setting gets its own step-4 test and its own verdict, at alpha = 0.025 each
  (Bonferroni over the two settings). If neither passes, the QQP population cannot test the
  hypothesis, reported as such.
  Amendment 2026-11-14 (after an independent review, before any stage-A result was seen):
  groups also merge items whose sentences share a multiset of words (PAWS word-swap families).
"""

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import feather_eval as fe  # noqa: E402

fe.REPO_TMPL = "connectivity/bert_ft_qqp-{i}"   # model_source / purge_from_cache read this
QQP_LABELS = {"not_duplicate": 0, "duplicate": 1}
DUP = 1
N_QQP = 10000
PAWSQ_FILE = "paws_eval@34110steps_bert_ft_qqp-{i}.json"
GATE_R, GATE_SD, GATE_RANGE = 0.80, 3.0, 0.10


def load_qqp(n=N_QQP):
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", "qqp", split="validation")
    idx = np.sort(np.random.default_rng(0).choice(len(ds), n, replace=False))
    sub = ds.select(idx.tolist())
    return {"premise": sub["question1"], "hypothesis": sub["question2"], "label": np.array(sub["label"])}


def load_paws_wiki():
    from datasets import load_dataset
    for name in ("google-research-datasets/paws", "paws"):
        try:
            ds = load_dataset(name, "labeled_final", split="test"); break
        except Exception as e:  # noqa: BLE001
            err = e
    else:
        raise err
    return {"premise": ds["sentence1"], "hypothesis": ds["sentence2"], "label": np.array(ds["label"])}


def load_pawsq(i):
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(fe.REPO_TMPL.format(i=i), PAWSQ_FILE.format(i=i))
    d = json.load(open(p))
    assert {k.lower(): int(v) for k, v in d["label_dict"].items()} == QQP_LABELS, d["label_dict"]
    prem, hyp, lab, logit, keep = [], [], [], [], []
    for s in d["samples"]:
        pr = fe.split_stored_sentence(s["sentence"])
        if pr is None:
            continue
        prem.append(pr[0]); hyp.append(pr[1]); lab.append(int(s["label"])); logit.append(s["logits"])
    return {"premise": prem, "hypothesis": hyp, "label": np.array(lab),
            "stored": np.array(logit, np.float32)}


def run(tok, model, P, H, device, bs=256):
    import torch
    n = len(P); order = np.argsort([len(a) + len(b) for a, b in zip(P, H)])
    out = np.empty((n, model.config.num_labels), np.float32)
    with torch.inference_mode():
        for s in range(0, n, bs):
            ix = order[s:s + bs]
            enc = tok([P[j] for j in ix], [H[j] for j in ix], truncation=True, max_length=fe.MAX_LEN,
                      padding=True, return_tensors="pt").to(device)
            out[ix] = model(**enc).logits.float().cpu().numpy()
    return out


def cmd_run(args):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    device = fe.pick_device(args.device)
    out = Path(args.out_dir); (out / "logits").mkdir(parents=True, exist_ok=True)
    qqp = load_qqp(); pw = load_paws_wiki()
    ref = load_pawsq(0)
    np.savez(out / "meta.npz", qqp_label=qqp["label"], pawsw_label=pw["label"], pawsq_label=ref["label"])
    print(f"device={device}  QQP n={len(qqp['label'])}  PAWS-QQP n={len(ref['label'])} "
          f"(non-dup {(ref['label'] == 0).sum()})  PAWS-Wiki n={len(pw['label'])} (non-dup {(pw['label'] == 0).sum()})")
    for k, i in enumerate(fe.parse_models(args.models)):
        f = out / "logits" / f"model_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            pq = load_pawsq(i)
            same = pq["premise"] == ref["premise"] and pq["hypothesis"] == ref["hypothesis"]
            path, sha = fe.model_source(i, args)
            tok = AutoTokenizer.from_pretrained(path)
            model = BertForSequenceClassification.from_pretrained(path).to(device).eval()
            assert model.config.num_labels == 2
            ql = run(tok, model, qqp["premise"], qqp["hypothesis"], device)
            pql = run(tok, model, ref["premise"], ref["hypothesis"], device)
            pwl = run(tok, model, pw["premise"], pw["hypothesis"], device)
        except Exception as e:  # noqa: BLE001
            print(f"[{k+1}] model {i}: FAILED {type(e).__name__}: {str(e)[:300]}"); continue
        tmp = f.with_suffix(".tmp.npz")
        np.savez(tmp, qqp_logits=ql, pawsq_logits=pql, pawsw_logits=pwl,
                 pawsq_stored=pq["stored"] if same else np.zeros(0), same_items=same, sha=sha)
        tmp.replace(f)
        agree = (pql.argmax(1) == pq["stored"].argmax(1)).mean() if same else float("nan")
        print(f"[{k+1}] model {i}: QQP acc {(ql.argmax(1) == qqp['label']).mean():.4f}  "
              f"PAWS-QQP non-dup {(pql.argmax(1)[ref['label'] == 0] != DUP).mean():.3f}  "
              f"PAWS-Wiki non-dup {(pwl.argmax(1)[pw['label'] == 0] != DUP).mean():.3f}  "
              f"stored-agree {agree:.4f}  ({time.time()-t0:.0f}s)")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
        if args.purge:
            fe.purge_from_cache(i)


def sentence_groups(P, H):
    """Union-find: items that share a sentence, OR a sentence with the same multiset of words (PAWS
    builds pairs by swapping words, so families share a bag of words), form one group."""
    import re
    parent = list(range(len(P))); first = {}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for k, (p, h) in enumerate(zip(P, H)):
        for s in (p, h):
            toks = re.findall(r"\w+", s.lower())
            for key in (" ".join(toks), "BAG:" + " ".join(sorted(toks))):
                if key in first:
                    parent[find(k)] = find(first[key])
                else:
                    first[key] = k
    return np.array([find(k) for k in range(len(P))])


def cmd_groups(args):
    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    pw = load_paws_wiki(); pq = load_pawsq(0)
    np.savez(out / "paws_groups.npz", pawsw=sentence_groups(pw["premise"], pw["hypothesis"]),
             pawsq=sentence_groups(pq["premise"], pq["hypothesis"]))
    g = np.load(out / "paws_groups.npz")
    print(f"PAWS-Wiki: {len(g['pawsw'])} items in {len(set(g['pawsw']))} groups; "
          f"PAWS-QQP: {len(g['pawsq'])} items in {len(set(g['pawsq']))} groups")


def gate_one(C, rng, groups, n_splits=1000):
    """C: models x items correctness; groups: group id per item (None = random item split)."""
    from scipy.stats import pearsonr
    y = C.mean(1); p = y.mean(); n = C.shape[1]
    rs = []
    ug = np.unique(groups) if groups is not None else None
    for _ in range(n_splits):
        if ug is None:
            a = np.zeros(n, bool); a[rng.permutation(n)[: n // 2]] = True
        else:
            a = np.isin(groups, rng.permutation(ug)[: len(ug) // 2])
        rs.append(pearsonr(C[:, a].mean(1), C[:, ~a].mean(1))[0])
    sd, bs = y.std(ddof=1), np.sqrt(p * (1 - p) / n)
    r = float(np.mean(rs))
    ok = r >= GATE_R and sd >= GATE_SD * bs and y.max() - y.min() >= GATE_RANGE
    return dict(n_items=int(n), n_groups=int(len(ug)) if ug is not None else int(n),
                mean=float(p), sd=float(sd), sd_ratio=float(sd / bs),
                min=float(y.min()), max=float(y.max()), split_half_r=r,
                r_lo=float(np.percentile(rs, 2.5)), PASS=bool(ok))


def cmd_gate(args):
    from scipy.stats import pearsonr
    out = Path(args.out_dir); meta = np.load(out / "meta.npz")
    files = sorted((out / "logits").glob("model_*.npz"))
    Z = [np.load(f) for f in files]
    ids = [int(f.stem.split("_")[1]) for f in files]
    qacc = np.array([(z["qqp_logits"].argmax(1) == meta["qqp_label"]).mean() for z in Z])
    agree = [float((z["pawsq_logits"].argmax(1) == z["pawsq_stored"].argmax(1)).mean())
             for z in Z if z["pawsq_stored"].size]
    gf = out / "paws_groups.npz"
    if not gf.exists():
        sys.exit("paws_groups.npz missing: run `python fix1_qqp.py groups` first (needs internet).")
    G = np.load(gf)
    rng = np.random.default_rng(0)
    res = {}
    for name, key, lab, gk in [("PAWS-QQP", "pawsq_logits", meta["pawsq_label"], "pawsq"),
                               ("PAWS-Wiki", "pawsw_logits", meta["pawsw_label"], "pawsw")]:
        C = np.array([z[key].argmax(1)[lab == 0] != DUP for z in Z])
        res[name] = gate_one(C, rng, G[gk][lab == 0])
        res[name]["random_split_r"] = gate_one(C, rng, None, 200)["split_half_r"]
        res[name]["r_with_qqp_acc"] = float(pearsonr(qacc, C.mean(1))[0])
    passing = [k for k in ("PAWS-QQP", "PAWS-Wiki") if res[k]["PASS"]]
    L = [f"FIX 1 STAGE A: QQP population ({len(ids)} models)", "",
         f"QQP validation accuracy (10k subsample): {qacc.min():.4f} - {qacc.max():.4f}",
         f"Reproduction of stored PAWS-QQP predictions: argmax agreement min {min(agree):.4f}, "
         f"median {np.median(agree):.4f} over {len(agree)} models" if agree else
         "Reproduction: stored items differ across repos (not compared)", ""]
    for k, v in res.items():
        L.append(f"{k:10s} non-paraphrase accuracy: n={v['n_items']} in {v['n_groups']} groups  mean {v['mean']:.3f}  "
                 f"range {v['min']:.3f}-{v['max']:.3f}  SD {v['sd']:.4f} (x{v['sd_ratio']:.1f} sampling)  "
                 f"group split-half r {v['split_half_r']:.3f} [{v['r_lo']:.2f}] (random-item split {v['random_split_r']:.3f})  "
                 f"r(QQP acc) {v['r_with_qqp_acc']:+.2f}  -> {'PASS' if v['PASS'] else 'FAIL'}")
    L += ["", f"SETTINGS THAT PASS (each tested separately): {', '.join(passing)}" if passing else
          "VERDICT: no reliable OOD outcome in the QQP population; it cannot test the hypothesis."]
    rep = "\n".join(L); print(rep)
    (out / "gate_report.txt").write_text(rep + "\n")
    (out / "gate_summary.json").write_text(json.dumps({"results": res, "passing": passing}, indent=2))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["run", "gate", "groups"]:
        p = sp.add_parser(name); p.add_argument("--out_dir", default="qqp_out")
        if name == "run":
            p.add_argument("--models", nargs="+", default=["all"])
            p.add_argument("--device", default="auto")
            p.add_argument("--local_tmpl", default=None)
            p.add_argument("--purge", action="store_true")
    args = ap.parse_args()
    {"run": cmd_run, "gate": cmd_gate, "groups": cmd_groups}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile qqp_step4.py
#!/usr/bin/env python3
"""
Fix 1, stage B: the step-4 test, unchanged, on the QQP population (run only if stage A's gate
named a primary outcome).

Everything is step 4's design, transplanted: a fresh 70/30 split (seed 2026), the same feature
families computed from QQP validation inputs only (never PAWS), the same models and the same
pre-registered decision rule. Task-specific changes, fixed in advance:
  - the "shortcut-prone" label is duplicate (index 1), playing entailment's role;
  - log-odds are 2-class: logit[duplicate] - logit[not_duplicate];
  - the probe: 256 high-overlap + 256 other QQP validation pairs (same overlap measure);
  - outcome: --outcome PAWS-QQP or PAWS-Wiki (non-paraphrase accuracy); each is a DISTINCT
    evaluation setting with its own verdict, allowed only if it passed stage A's gate.

PRE-REGISTERED AMENDMENT (2026-11-14, after an independent review of the MNLI study, before any
QQP feature or test result exists). The MNLI primary contrast was degenerate: the stacker gave the
internal blocks zero weight, so dR2 was exactly 0 with a [0, 0] "CI". For QQP the primary test is:
  b = out-of-sample prediction from ALL BASELINES (fitted on the 70 dev models),
  i = out-of-sample prediction from INTERNAL (fitted on the 70 dev models);
  on the 30 test models, the partial correlation r_p of y with i controlling for b.
  Inference: one-sided permutation test (10,000 permutations of i), alpha = 0.025 per PAWS setting
  (Bonferroni over the two settings). Interval: 95% Fisher-z CI for r_p (df = n - 4).
  GO: p < 0.025 AND r_p >= 0.30 (an increment of at least ~9% of the variance baselines leave).
  NULL-CONSISTENT: otherwise. A claim of "no meaningful increment" is made only if the 90% CI of
  r_p lies inside [-0.30, 0.30]; with n = 30 this is not reachable, which is stated in advance.
  The stacked dR2 of step 4 is still reported, labelled as descriptive.
  Ridge grid extended to 1e-8 .. 1e4 (the MNLI fits hit the lower boundary of 1e-4).
  Exploratory companion: repeated 10-fold CV over all 100 QQP models, same statistic.

Subcommands: split | extract (GPU, resumable, --purge) | analyze [--final]
"""

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import feather_eval as fe     # noqa: E402
import feather_step4 as s4    # noqa: E402
import fix1_qqp as q          # noqa: E402  (sets fe.REPO_TMPL to the QQP models)

fe.ENT = q.DUP


def logodds_2(logits):
    import torch
    t = torch.as_tensor(logits)
    return t[..., 1] - t[..., 0]


s4.ent_logodds = logodds_2     # extract_one looks this up at call time
s4.ALPHAS = np.logspace(-8, 4, 37)   # amended grid (MNLI fits hit the old lower boundary)
R_P_MIN, ALPHA_SETTING = 0.30, 0.025


def partial_r_test(y, b, i, n_perm=10000, seed=0):
    """Partial correlation of y and i given b; one-sided permutation p; 95% and 90% Fisher-z CIs."""
    def pr(yv, iv):
        res = lambda v: v - np.poly1d(np.polyfit(b, v, 1))(b)  # noqa: E731
        ry, ri = res(yv), res(iv)
        return float(ry @ ri / np.sqrt((ry @ ry) * (ri @ ri)))
    r = pr(y, i); rng = np.random.default_rng(seed)
    perm = np.array([pr(y, rng.permutation(i)) for _ in range(n_perm)])
    p = float((np.sum(perm >= r) + 1) / (n_perm + 1))
    z, se = np.arctanh(r), 1 / np.sqrt(len(y) - 4)
    ci95 = np.tanh([z - 1.96 * se, z + 1.96 * se]); ci90 = np.tanh([z - 1.645 * se, z + 1.645 * se])
    return r, p, ci95, ci90


def cmd_split(args):
    s4.cmd_split(args)


def cmd_extract(args):
    import torch
    from transformers import BertModel
    device = fe.pick_device(args.device)
    out = Path(args.out_dir); fdir = out / "features"; fdir.mkdir(parents=True, exist_ok=True)
    meta = np.load(out / "meta.npz")
    qqp = q.load_qqp()
    assert np.array_equal(qqp["label"], meta["qqp_label"]), "QQP subsample differs from stage A"
    probe = s4.build_probe(qqp, out)
    pt = ([qqp["premise"][j] for j in probe["idx"]], [qqp["hypothesis"][j] for j in probe["idx"]])
    pre = {k: v.detach().cpu() for k, v in BertModel.from_pretrained(args.pretrained).state_dict().items()}
    models = fe.parse_models(args.models)
    for k, i in enumerate(models):
        f = fdir / f"feat_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            res = s4.extract_one(i, args, device, pt, probe["is_ho"], pre)
        except Exception as e:  # noqa: BLE001
            print(f"[{k+1}/{len(models)}] model {i}: FAILED {type(e).__name__}: {str(e)[:300]}")
            if device == "cuda":
                torch.cuda.empty_cache()
            continue
        tmp = f.with_suffix(".tmp.npz"); np.savez(tmp, **res); tmp.replace(f)
        print(f"[{k+1}/{len(models)}] model {i}: ok  max|abl_ho|={np.abs(res['abl_ho']).max():.3f}  "
              f"fp16 err={res['amp_max_err']:.1e}  ({time.time()-t0:.0f}s)")
        if args.purge:
            fe.purge_from_cache(i)


def build_blocks_qqp(out, ids, prim):
    meta = np.load(out / "meta.npz")
    gate = json.loads((out / "gate_summary.json").read_text())
    assert prim in gate["passing"], f"{prim} did not pass stage A's gate; it cannot be analysed"
    key, lab = {"PAWS-QQP": ("pawsq_logits", meta["pawsq_label"]),
                "PAWS-Wiki": ("pawsw_logits", meta["pawsw_label"])}[prim]
    fdir = out / "features"
    ho = np.load(fdir / "mnli_overlap.npy") >= s4.HO_THRESH     # overlap on the QQP subsample
    ql = meta["qqp_label"]
    y, idsum, iditems, wts, sk, attn, abl, reps, acc = [], [], [], [], [], [], [], [], []
    for i in ids:
        z = np.load(out / "logits" / f"model_{i:03d}.npz")
        y.append((z[key].argmax(1)[lab == 0] != q.DUP).mean())
        lg = z["qqp_logits"]; pr = np.exp(lg - lg.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
        corr = lg.argmax(1) == ql
        hn, he = ho & (ql == 0), ho & (ql == 1)
        acc.append(corr.mean())
        idsum.append([corr.mean(), corr[ql == 0].mean(), corr[ql == 1].mean(), pr.max(1).mean(),
                      pr[:, 1].mean(), corr[ho].mean(), corr[hn].mean(), pr[hn, 1].mean(), pr[he, 1].mean()])
        iditems.append(lg[ho, 1] - lg[ho, 0])
        f = np.load(fdir / f"feat_{i:03d}.npz")
        wts.append(np.concatenate([f["delta_norm"], f["delta_rel"], f["sketch"], f["clf"]])); sk.append(f["sketch"])
        attn.append(np.concatenate([f["attn_match"].ravel(), f["attn_prem"].ravel()]))
        abl.append(np.concatenate([f["abl_ho"].ravel(), f["abl_rand"].ravel()]))
        reps.append(f["rep"])
    reps = np.array(reps)
    cka = np.mean([s4.linear_cka_kernel(reps[:, l]) for l in range(reps.shape[1])], 0)
    blocks = {"MNLI-acc": s4.linear_kernel_block(np.array(acc)[:, None]),   # = QQP accuracy here
              "ID-summary": s4.linear_kernel_block(idsum),
              "ID-overlap-items": s4.linear_kernel_block(iditems),
              "Weights": s4.linear_kernel_block(wts),
              "Weight-geometry": s4.rbf_distance_block(sk),
              "Repr-CKA": s4.fixed_kernel_block(cka),
              "Attn-stats": s4.linear_kernel_block(attn),
              "Head-ablation": s4.linear_kernel_block(abl)}
    return blocks, np.array(y), prim, int(ho.sum())


def cmd_analyze(args):
    out = Path(args.out_dir)
    split = json.loads((out / "split.json").read_text())
    have = {int(p.stem.split("_")[1]) for p in (out / "features").glob("feat_*.npz")}
    dev = [i for i in split["dev"] if i in have]; test = [i for i in split["test"] if i in have]
    ids = dev + test
    blocks, y, prim, n_ho = build_blocks_qqp(out, ids, args.outcome)
    dev_idx, test_idx = np.arange(len(dev)), np.arange(len(dev), len(ids))
    L = [f"QQP REPLICATION OF STEP 4 ({'FINAL: test models' if args.final else 'development CV'})",
         f"models: {len(dev)} dev, {len(test)} test; outcome: {prim} non-paraphrase accuracy; "
         f"high-overlap QQP items: {n_ho}", "('MNLI-acc only' = QQP accuracy only in this population)", ""]
    preds = {}
    for sname, names in s4.SETS.items():
        if args.final:
            preds[sname] = s4.predict_set(blocks, names, dev_idx, test_idx, y)
            L.append(f"{sname:32s} test R2 {s4.r2(y[test_idx], preds[sname]):7.3f}")
        else:
            preds[sname], r2s = s4.cv_oof(blocks, names, dev_idx, y, args.n_rep, args.seed)
            L.append(f"{sname:32s} CV R2 {r2s.mean():7.3f}  [{r2s.min():.3f}, {r2s.max():.3f}]")
    yy = y[test_idx] if args.final else y[dev_idx]
    A, B = "ALL BASELINES + INTERNAL", "ALL BASELINES"
    d = s4.r2(yy, preds[A]) - s4.r2(yy, preds[B])
    r, p, ci95, ci90 = partial_r_test(yy, preds[B], preds["INTERNAL (attn+ablation)"])
    go = p < ALPHA_SETTING and r >= R_P_MIN
    equiv = ci90[0] > -R_P_MIN and ci90[1] < R_P_MIN
    L += ["", f"PRIMARY (amended): partial r of outcome with INTERNAL given ALL BASELINES = {r:+.3f}  "
              f"95% CI [{ci95[0]:+.3f}, {ci95[1]:+.3f}]  permutation p (one-sided) = {p:.4f}",
          f"  -> {'GO' if go else 'NULL-CONSISTENT'} at alpha {ALPHA_SETTING} and r_p >= {R_P_MIN}"
          f"{'; equivalence (90% CI within +/-0.30) shown' if equiv else '; equivalence NOT shown (90% CI exceeds +/-0.30)'}",
          f"  descriptive: stacked dR2 = {d:+.3f} (can be exactly 0 by construction; not used for the verdict)",
          f"VERDICT ({'binding' if args.final else 'provisional, development CV'}): {'GO' if go else 'NULL-CONSISTENT'}"]
    rep = "\n".join(L); print(rep)
    (out / f"qqp_step4_{args.outcome}_{'final' if args.final else 'dev'}.txt").write_text(rep + "\n")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["split", "extract", "analyze"]:
        p = sp.add_parser(name); p.add_argument("--out_dir", default="qqp_out"); p.add_argument("--seed", type=int, default=0)
        if name == "split":
            p.add_argument("--force", action="store_true")
        if name == "extract":
            p.add_argument("--models", nargs="+", default=["all"]); p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=64); p.add_argument("--local_tmpl", default=None)
            p.add_argument("--pretrained", default=s4.PRETRAINED); p.add_argument("--purge", action="store_true")
            p.add_argument("--fp32_ablation", action="store_true")
        if name == "analyze":
            p.add_argument("--final", action="store_true"); p.add_argument("--n_rep", type=int, default=20)
            p.add_argument("--outcome", required=True, choices=["PAWS-QQP", "PAWS-Wiki"])
    args = ap.parse_args()
    {"split": cmd_split, "extract": cmd_extract, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
# 5. Environment check
import torch, transformers, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "Enable a GPU accelerator in Session options."
print(torch.cuda.get_device_name(0))

In [ ]:
# 6. Optional Hugging Face token
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; downloading anonymously")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [ ]:
# 7. Bring in stage A results (meta.npz, logits/, gate_summary.json)
import glob, os, shutil, zipfile, json
hits = glob.glob("/kaggle/input/**/qqp_out/gate_summary.json", recursive=True)
if not hits:
    for z in glob.glob("/kaggle/input/**/fix14_out.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/working/_in"); break
    hits = glob.glob("/kaggle/working/_in/**/qqp_out/gate_summary.json", recursive=True)
assert hits, "Stage A output not found: Add Input -> the fixes 1+4 notebook output"
src = os.path.dirname(hits[0]); os.makedirs("qqp_out", exist_ok=True)
for f in ["meta.npz", "gate_summary.json", "gate_report.txt", "paws_groups.npz"]:
    shutil.copy(os.path.join(src, f), "qqp_out/")
if not os.path.exists("qqp_out/logits"):
    shutil.copytree(os.path.join(src, "logits"), "qqp_out/logits")
passing = json.load(open("qqp_out/gate_summary.json"))["passing"]
print(open("qqp_out/gate_report.txt").read())
assert passing, "No PAWS setting passed the gate: stage B should not be run."
print("settings to test (separately):", passing)

In [ ]:
# 8. Freeze the 70/30 split for this population
!python qqp_step4.py split

In [ ]:
# 9. Timing test on 2 models
!python qqp_step4.py extract --models 0 1 --purge 2>&1 | grep -v "Loading weights"

In [ ]:
# 10. All 100 models (resumable)
!python qqp_step4.py extract --models all --purge 2>&1 | grep -v "Loading weights"

In [ ]:
# 11. Development analysis, one per passing setting (provisional; test models are scored once, later)
for s in passing:
    !python qqp_step4.py analyze --outcome {s}

In [ ]:
# 12. Package
!zip -qr qqp_step4_out.zip qqp_out/split.json qqp_out/features qqp_out/qqp_step4_*_dev.txt && ls -lh qqp_step4_out.zip